# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables

load_dotenv()


True

In [2]:
import dask.dataframe as dd

c:\Users\Sima\anaconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [14]:
import os
from glob import glob
import pandas as pd

# Load environment variables
load_dotenv()

# Retrieve the PRICE_DATA variable
price_data_path = os.getenv("PRICE_DATA")

# Find all parquet files in the directory
parquet_files = glob(os.path.join(price_data_path, "**/*.parquet"), recursive = True)

# Check the parquet files
print("Parquet Files Found")

#Show the first 5 file paths
parquet_files[:5]


Parquet Files Found


['../../05_src/data/prices\\AAPL\\AAPL_2000\\part.0.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2000\\part.1.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2001\\part.0.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2001\\part.1.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2002\\part.0.parquet']

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [15]:
# Load all parquet files into a Dask DataFrame
dd_px = dd.read_parquet(parquet_files).set_index("Ticker")

# Add lags for Close and Adj_Close
dd_shift = dd_px.groupby('Ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1=x['Close'].shift(1),
        Adj_Close_lag_1=x['Adj Close'].shift(1) if "Adj Close" in x.columns else None
    )
)

# Add returns based on Close
dd_feat = dd_shift.assign(
    Returns=lambda x: x['Close'] / x['Close_lag_1'] - 1,
    hi_lo_range=lambda x: x['High'] - x['Low']
)

# Assign result to dd_feat 
dd_feat = dd_shift.assign(Returns = lambda x: x['Close']/x['Close_lag_1'] - 1)



c:\Users\Sima\anaconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
C:\Users\Sima\AppData\Local\Temp\ipykernel_20780\69372828.py:5: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = dd_px.groupby('Ticker', group_keys=False).apply(


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [16]:
# Convert the Dask DataFrame to a Pandas DataFrame
df_pandas = dd_feat.compute()

# Add a 10-day moving average of returns
df_pandas["returns_ma_10"] = df_pandas["Returns"].rolling(10).mean()

# Print sample output for verification
print(df_pandas.head())


             Date  Adj Close     Close      High       Low      Open  \
Ticker                                                                 
AAPL   2000-01-03   0.843076  0.999442  1.004464  0.907924  0.936384   
AAPL   2000-01-04   0.771996  0.915179  0.987723  0.903460  0.966518   
AAPL   2000-01-05   0.783293  0.928571  0.987165  0.919643  0.926339   
AAPL   2000-01-06   0.715509  0.848214  0.955357  0.848214  0.947545   
AAPL   2000-01-07   0.749401  0.888393  0.901786  0.852679  0.861607   

             Volume  Year  Close_lag_1  Adj_Close_lag_1   Returns  \
Ticker                                                              
AAPL    535796800.0  2000          NaN              NaN       NaN   
AAPL    512377600.0  2000     0.999442         0.843076 -0.084310   
AAPL    778321600.0  2000     0.915179         0.771996  0.014633   
AAPL    767972800.0  2000     0.928571         0.783293 -0.086538   
AAPL    460734400.0  2000     0.848214         0.715509  0.047369   

        ret

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

No, it was not necessary to convert the Dask DataFrame (dd_feat) into a Pandas DataFrame (df_pandas) to calculate the 10-day moving average. Dask supports rolling window operations through its built-in .rolling() method.

+ Would it have been better to do it in Dask? Why?

Yes, it would have been better to compute the moving average in Dask rather than converting it to Pandas in order to improve memory efficiency and scalability.


(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.